#Profit calculation for A1

Imports and setup -
This cell imports all the fundamental Python libraries used in this part of the project.
os, json, math, and sys handle system and file operations, numpy provides numerical functions, and pandas is used for tabular data manipulation.
These imports ensure that the notebook can be executed independently, even when run standalone in Google Colab.

In [1]:
# ============================================================
# Profit optimization for baseline (A0) vs extended model (A1)
# Colab-ready, standalone block
# ============================================================

# --- Imports
import os, json, math, sys
import numpy as np
import pandas as pd

Mount Google Drive (Colab) – Mounts the user’s Google Drive at /content/drive so that result files (e.g. profit_comparison.csv) can be saved and accessed directly from Google Drive. If the notebook is not running in Colab, the mount is skipped.

In [2]:
# --- Mount Google Drive (Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✔ Google Drive mounted at /content/drive")
except ImportError:
    # Not running in Google Colab
    print("Google Drive mounting skipped (not running in Colab).")

Mounted at /content/drive
✔ Google Drive mounted at /content/drive


Profit model parameters -
Defines the constants for the analytical profit model:

𝛼=1000,
𝜀=3,
𝑏=12,
𝑘=0.3, and
𝑚ₒₚₛ=0.03.

These values determine the demand elasticity, operational costs, and profit structure.
They are identical to those used for the baseline model (A₀), which ensures a fair comparison with the extended model (A₁).

In [3]:
# --- Parameters (same as baseline per assignment)
alpha = 1000
epsilon = 3
b = 12
k = 0.3
m_ops = 0.03

Extended model accuracy (A₁) -
Stores the classification accuracy of the extended model,
𝐴₁=0.5797.

This value was obtained from the earlier model training phase and is used here to compute optimal margins and total profit.

In [4]:
# --- Extended model accuracy (given)
A1 = 0.5797

Function for calculating optimal metrics for a given accuracy

Description:
Defines the function optimal_profit_metrics(A), which analytically computes all key economic metrics for a given model accuracy
𝐴:

𝑐(𝐴)=𝑚ₒₚₛ+𝑘(1−𝐴) - total cost

𝑚_𝑝𝑟𝑜𝑓𝑖𝑡∗*=𝑐(𝐴)/(𝜀−1) – optimal profit margin

𝑚*=𝜀⋅𝑚_𝑝𝑟𝑜𝑓𝑖𝑡* – total optimal margin

𝑉(𝑚*)=𝛼⋅(𝑚*)−𝜀 – demand function

Π*=𝑉(𝑚*)⋅𝑚_𝑝𝑟𝑜𝑓𝑖𝑡*⋅𝑏 – optimal total profit

The function returns these values as a dictionary for easy comparison between A₀ and A₁.

In [5]:
# --- Helper: compute optimal metrics for given accuracy A
def optimal_profit_metrics(A, alpha=alpha, epsilon=epsilon, b=b, k=k, m_ops=m_ops):
    """
    Returns dict with:
      - c(A) = m_ops + k(1 - A)
      - m_profit* = c / (epsilon - 1)
      - m_total*  = epsilon * m_profit*
      - V(m*)     = alpha * (m_total*)^(-epsilon)
      - Pi*       = V * m_profit* * b
    """
    c = m_ops + k*(1 - float(A))
    m_profit_star = c / (epsilon - 1)
    m_total_star  = epsilon * m_profit_star
    V = alpha * (m_total_star ** (-epsilon))
    Pi_star = V * m_profit_star * b
    return {
        "A": float(A),
        "c(A)": c,
        "m_profit*": m_profit_star,
        "m_total*": m_total_star,
        "V(m*)": V,
        "Pi*": Pi_star,
    }

The 𝐴₀=0.554 accuracy is loaded and stored in the variable A₀ for further calculations.

In [6]:
# Baseline accuracy set manually
A0 = 0.554  # baseline accuracy

Compute metrics for A₀ and A₁ - Calls the optimal_profit_metrics() function for both the baseline (A₀) and extended (A₁) models.

The results — optimal margins, demand, and total profits — are stored in metrics_A₀ and metrics_A₁ for comparison.

In [7]:
# --- Compute metrics
metrics_A0 = optimal_profit_metrics(A0)
metrics_A1 = optimal_profit_metrics(A1)

Combine results and compute ΔΠ -
Builds a summary DataFrame that displays all computed values for both models:

𝐴, 𝑐(𝐴), 𝑚_𝑝𝑟𝑜𝑓𝑖𝑡*, 𝑚_𝑡𝑜𝑡𝑎𝑙*,
𝑉(𝑚*), and Π*.

Additionally, it calculates the difference in optimal profit between the models:

ΔΠ=Π(𝐴₁)−Π(𝐴₀),

which quantifies the improvement achieved by the extended model.

In [8]:
# --- Summary table + ΔΠ
delta = metrics_A1["Pi*"] - metrics_A0["Pi*"]
summary = pd.DataFrame([
    {
        "Model": "Baseline (A0)",
        "A": metrics_A0["A"],
        "c(A)": metrics_A0["c(A)"],
        "m_profit*": metrics_A0["m_profit*"],
        "m_total*": metrics_A0["m_total*"],
        "V(m*)": metrics_A0["V(m*)"],
        "Pi*": metrics_A0["Pi*"],
    },
    {
        "Model": "Extended (A1)",
        "A": metrics_A1["A"],
        "c(A)": metrics_A1["c(A)"],
        "m_profit*": metrics_A1["m_profit*"],
        "m_total*": metrics_A1["m_total*"],
        "V(m*)": metrics_A1["V(m*)"],
        "Pi*": metrics_A1["Pi*"],
    }
])

Formatted output of results -
Uses Pandas to print the comparison table in a readable format and display the computed profit difference ΔΠ.

This output represents the main summary of this analytical step, providing a clear economic interpretation of the model improvement.

In [9]:
# Display nicely
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
print("\n=== Profit Optimization Summary ===")
print(summary.to_string(index=False))
print(f"\nΔΠ = Π(A1) − Π(A0) = {delta:,.2f}")


=== Profit Optimization Summary ===
        Model        A     c(A)  m_profit*  m_total*         V(m*)           Pi*
Baseline (A0) 0.554000 0.163800   0.081900  0.245700 67,419.345258 66,259.732519
Extended (A1) 0.579700 0.156090   0.078045  0.234135 77,911.395098 72,967.137965

ΔΠ = Π(A1) − Π(A0) = 6,707.41


Save results to CSV file -
Saves the summary table as profit_comparison.csv.

The file includes all calculated metrics and the ΔΠ value.

It can be committed to the repository or easily saved to Google Drive for documentation and further analysis.

In [10]:
# Save an artifact you can commit
out_path = "./profit_comparison.csv"
summary.assign(delta_Pi_vs_prev=[np.nan, delta]).to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")


Saved: ./profit_comparison.csv


Download the results file -
Allows the user to download profit_comparison.csv directly from Colab to their local machine using the google.colab.files module.

This step is optional and provides an alternative way to access the results if Google Drive integration is not used.

In [11]:
from google.colab import files
files.download("profit_comparison.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

####Summary of Profit calculation for A₁

In this section, we analytically computed the optimal profit model for both the baseline and extended classifiers.

Using the shared parameters
𝛼=1000,
𝜀=3,
𝑏=12,
𝑘=0.3, and
𝑚ₒₚₛ=0.03,
the optimal profit, margin, and demand were calculated for each model’s accuracy (
𝐴₀=0.554,
𝐴₁=0.5797).

The extended model achieved a higher optimal profit of
Π*(𝐴₁)=72,967.14
compared to the baseline
Π*(𝐴₀)=66,259.73
, representing an improvement of
ΔΠ=6,707.41.

This positive difference demonstrates the extended model’s economic benefit over the baseline.

All computed results are saved to profit_comparison.csv, which serves as an artifact for documentation and comparison across team deliverables.